In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments, DataCollatorForTokenClassification
from datasets import Dataset
import torch
import numpy as np

In [ ]:
label_list = ["O", "B-NAME", "I-NAME", "B-DIMENSION", "I-DIMENSION"]

In [ ]:
def encode_dataset(examples, tokenizer):
    """Encode the dataset for the model with fixed length sequences"""
   
    max_length = 64  
    
   
    tokenized_inputs = tokenizer(
        examples["text"],
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors=None  # Return list instead of tensors
    )
    
   
    labels = []
    for i in range(len(examples["text"])):
    
        tokens = tokenizer.convert_ids_to_tokens(tokenized_inputs["input_ids"][i])
        current_labels = [-100] * max_length
        
        for j, token in enumerate(tokens):
            if j >= max_length:
                break
                

            if token in ["[CLS]", "[SEP]", "[PAD]"]:
                current_labels[j] = -100
                continue
                
            
            if any(c.isdigit() for c in token):
                if j > 0 and current_labels[j-1] == label_list.index("B-DIMENSION"):
                    current_labels[j] = label_list.index("I-DIMENSION")
                else:
                    current_labels[j] = label_list.index("B-DIMENSION")
         
            else:
                if j > 0 and current_labels[j-1] == label_list.index("B-NAME"):
                    current_labels[j] = label_list.index("I-NAME")
                else:
                    current_labels[j] = label_list.index("B-NAME")
        
        labels.append(current_labels)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
def create_ner_model():
    model_name = "allenai/scibert_scivocab_uncased"
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=len(label_list),
        id2label={i: label for i, label in enumerate(label_list)},
        label2id={label: i for i, label in enumerate(label_list)}
    )
    
    return model, tokenizer

In [ ]:
def train_model(df, text_column='Short Text'):
    # Create dataset
    dataset = Dataset.from_dict({
        'text': df[text_column].astype(str).tolist()
    })
    
    
 
    model, tokenizer = create_ner_model()
    
    
    data_collator = DataCollatorForTokenClassification(tokenizer)
    
   
    encoded_dataset = dataset.map(
        lambda x: encode_dataset(x, tokenizer),
        batched=True,
        remove_columns=dataset.column_names
    )
    

    training_args = TrainingArguments(
        output_dir="./results",
        evaluation_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=8, 
        num_train_epochs=3,
        weight_decay=0.01,
        save_strategy="no",
        report_to="none",
        logging_steps=100
    )
    

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset,
        data_collator=data_collator,
    )
    

    trainer.train()
    
    return model, tokenizer

In [ ]:
def predict_entities(text, model, tokenizer):
    """Predict entities in new text"""
    max_length = 64  # Same as training
    
    # Tokenize input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding='max_length',
        max_length=max_length
    )
    
    
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)
    
    
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [label_list[p] for p in predictions[0]]
    
    # Extract entities
    entities = []
    current_entity = None
    
    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]", "[UNK]"]:
            continue
            
        if label.startswith("B-"):
            if current_entity:
                entities.append(current_entity)
            current_entity = {"type": label[2:], "text": token}
        elif label.startswith("I-") and current_entity:
            if not token.startswith("##"):
                current_entity["text"] += " " + token
            else:
                current_entity["text"] += token[2:]
        elif label == "O":
            if current_entity:
                entities.append(current_entity)
                current_entity = None
                
    if current_entity:
        entities.append(current_entity)
        
    return entities

In [ ]:
df=pd.read_excel("Test Data 2.xlsx")

In [ ]:
df.dropna(subset=['Short Text'],inplace=True)

In [ ]:
model, tokenizer = train_model(df)

In [ ]:
pip install 'accelerate>={ACCELERATE_MIN_VERSION}

In [ ]:
sample_text = "GROMMET EPDM ID25.4MM I/O PIPE"
entities = predict_entities(sample_text, model, tokenizer)
print(entities)

In [ ]:
import pickle

In [ ]:
with open('model.pkl', 'wb') as file:
    pickle.dump(model, file)

In [ ]:
with open('model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)